# 01 - Data Understanding and Cleaning

## Objective
The purpose of this notebook is to load raw Sephora product and review datasets,
clean and standardize them, and produce two main outputs:

1. `review_master.parquet` → review-level dataset
2. `product_master.parquet` → product-level dataset

## What we will do in this notebook
- Load raw datasets
- Inspect structure and schema
- Standardize columns
- Perform review-centric merge
- Fix data types
- Handle missing values
- Remove duplicates
- Preserve raw text
- Prepare dataset for NLP and recommendation modeling

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Data Sources

### Product Dataset
Contains product metadata such as:
- product_id
- product_name
- brand_name
- price
- ingredients
- category hierarchy
- availability flags

### Review Dataset
Contains user-generated reviews including:
- review_text
- rating
- recommendation flag
- helpfulness metrics
- user profile (skin_type, eye_color, etc.)

In [2]:
# Load product data
products = pd.read_csv(RAW_DIR / "product_info.csv")

# Load review data
review_files = [
    "reviews_0-250.csv",
    "reviews_250-500.csv",
    "reviews_500-750.csv",
    "reviews_750-1250.csv",
    "reviews_1250-end.csv"
]

reviews_list = []

for file in review_files:
    temp = pd.read_csv(
        RAW_DIR / file,
        low_memory=False,
        dtype={"author_id": "string"}  # prevent mixed dtype warning
    )
    reviews_list.append(temp)

reviews = pd.concat(reviews_list, ignore_index=True)

In [3]:
print("Products shape:", products.shape)
print("Reviews shape:", reviews.shape)

display(products.head())
display(reviews.head())

Products shape: (8494, 27)
Reviews shape: (1094411, 19)


,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,variation_value,variation_desc,ingredients,price_usd,value_price_usd,sale_price_usd,limited_edition,new,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
0,P473671,Fragrance Discovery Set,6342,19-69,6320,3.6364,11.0,NaN,NaN,NaN,NaN,"['Capri Eau de Parfum:', 'Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) D-Limonene, Linalool, Benzyl Salicylate, Ethylhexyl Methoxycinnamate, Butyl Methoxydibenzoylmethane, Ethylhexyl Salicy...",35.0,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Warm &Spicy Scent', 'Woody & Earthy Scent', 'Fresh Scent']",Fragrance,Value & Gift Sets,Perfume Gift Sets,0,NaN,NaN
1,P473668,La Habana Eau de Parfum,6342,19-69,3827,4.1538,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,NaN,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) Ethylhexyl Methoxycinnamate, Ethylhexyl Salicylate, Butyl Methoxydibenzoylmethane, Benzyl Alcohol, Benzyl Benzoate, Benzyl Cinnamate, Cinnamal...",195.0,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent', 'Warm &Spicy Scent']",Fragrance,Women,Perfume,2,85.0,30.0
2,P473662,Rainbow Bar Eau de Parfum,6342,19-69,3253,4.2500,16.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,NaN,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) D-Limonene, Ethylhexyl Methoxycinnamate, Butyl Methoxydibenzoylmethane, Ethylhexyl Salicylate, Hexyl Cinnamal Citronellol, Linalool, Coumarin,...",195.0,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent', 'Woody & Earthy Scent']",Fragrance,Women,Perfume,2,75.0,30.0
3,P473660,Kasbah Eau de Parfum,6342,19-69,3018,4.4762,21.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,NaN,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) Coumarin, Ethylhexyl Methoxycinnamate, Butyl Methoxydibenzoylmethane, Ethylhexyl Salicylate, D-Limonene, Eugenol, Linalool, Citronellol, Geran...",195.0,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent', 'Warm &Spicy Scent']",Fragrance,Women,Perfume,2,75.0,30.0
4,P473658,Purple Haze Eau de Parfum,6342,19-69,2691,3.2308,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,NaN,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) Octoxynol-10, D-Limonene, Ethylhexyl Methoxycinnamate, Butyl Methoxydibenzoylmethane, Ethylhexyl Salicylate, Geraniol, Linalool, Coumarin, Far...",195.0,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent', 'Woody & Earthy Scent']",Fragrance,Women,Perfume,2,75.0,30.0


,Unnamed: 0,author_id,rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,review_title,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd
0,0,1741593524,5,1.0,1.0,2,0,2,2023-02-01,I use this with the Nudestix “Citrus Clean Balm & Make-Up Melt“ to double cleanse and it has completely changed my skin (for the better). The make-up melt is oil based and removes all of your make...,Taught me how to double cleanse!,NaN,brown,dry,black,P504322,Gentle Hydra-Gel Face Cleanser,NUDESTIX,19.0
1,1,31423088263,1,0.0,NaN,0,0,0,2023-03-21,"I bought this lip mask after reading the reviews and the hype. Unfortunately, it did not meet my expectations as vaseline petroleum jelly works way better for me.",Disappointed,NaN,NaN,NaN,NaN,P420652,Lip Sleeping Mask Intense Hydration with Vitamin C,LANEIGE,24.0
2,2,5061282401,5,1.0,NaN,0,0,0,2023-03-21,My review title says it all! I get so excited to get into bed and apply this lip mask. I do see a difference because I suffer from dry cracked lips. I drink a lot of water and apply lip balm daily...,New Favorite Routine,light,brown,dry,blonde,P420652,Lip Sleeping Mask Intense Hydration with Vitamin C,LANEIGE,24.0
3,3,6083038851,5,1.0,NaN,0,0,0,2023-03-20,I’ve always loved this formula for a long time. I honestly don’t even use it for night time. I use it as an everyday lip balm. I love the texture. Gummy Bear is my second most favourite scent. Gra...,Can't go wrong with any of them,NaN,brown,combination,black,P420652,Lip Sleeping Mask Intense Hydration with Vitamin C,LANEIGE,24.0
4,4,47056667835,5,1.0,NaN,0,0,0,2023-03-20,"If you have dry cracked lips, this is a must have. After a few weeks of use I have learned I will always have by my bedside. I thought it was a little expensive but a little goes a long way. It is...",A must have !!!,light,hazel,combination,NaN,P420652,Lip Sleeping Mask Intense Hydration with Vitamin C,LANEIGE,24.0


In [4]:
print(products.columns.tolist())
print(reviews.columns.tolist())

['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count', 'rating', 'reviews', 'size', 'variation_type', 'variation_value', 'variation_desc', 'ingredients', 'price_usd', 'value_price_usd', 'sale_price_usd', 'limited_edition', 'new', 'online_only', 'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category', 'secondary_category', 'tertiary_category', 'child_count', 'child_max_price', 'child_min_price']
['Unnamed: 0', 'author_id', 'rating', 'is_recommended', 'helpfulness', 'total_feedback_count', 'total_neg_feedback_count', 'total_pos_feedback_count', 'submission_time', 'review_text', 'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color', 'product_id', 'product_name', 'brand_name', 'price_usd']


In [5]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8494 entries, 0 to 8493
Data columns (total 27 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   product_id          8494 non-null   object 
 1   product_name        8494 non-null   object 
 2   brand_id            8494 non-null   int64  
 3   brand_name          8494 non-null   object 
 4   loves_count         8494 non-null   int64  
 5   rating              8216 non-null   float64
 6   reviews             8216 non-null   float64
 7   size                6863 non-null   object 
 8   variation_type      7050 non-null   object 
 9   variation_value     6896 non-null   object 
 10  variation_desc      1250 non-null   object 
 11  ingredients         7549 non-null   object 
 12  price_usd           8494 non-null   float64
 13  value_price_usd     451 non-null    float64
 14  sale_price_usd      270 non-null    float64
 15  limited_edition     8494 non-null   int64  
 16  new   

In [6]:
reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1094411 entries, 0 to 1094410
Data columns (total 19 columns):
 #   Column                    Non-Null Count    Dtype  
---  ------                    --------------    -----  
 0   Unnamed: 0                1094411 non-null  int64  
 1   author_id                 1094411 non-null  string 
 2   rating                    1094411 non-null  int64  
 3   is_recommended            926423 non-null   float64
 4   helpfulness               532819 non-null   float64
 5   total_feedback_count      1094411 non-null  int64  
 6   total_neg_feedback_count  1094411 non-null  int64  
 7   total_pos_feedback_count  1094411 non-null  int64  
 8   submission_time           1094411 non-null  object 
 9   review_text               1092967 non-null  object 
 10  review_title              783757 non-null   object 
 11  skin_tone                 923872 non-null   object 
 12  eye_color                 884783 non-null   object 
 13  skin_type                 9

## 2. Column Selection

We keep only relevant columns for modeling and NLP.
We also ensure raw text is preserved.

In [7]:
review_cols = [
    "author_id", "rating", "is_recommended", "helpfulness",
    "total_feedback_count", "total_neg_feedback_count",
    "total_pos_feedback_count", "submission_time",
    "review_text", "review_title",
    "skin_tone", "eye_color", "skin_type", "hair_color",
    "product_id", "product_name", "brand_name", "price_usd"
]

product_cols = [
    "product_id", "product_name", "brand_id", "brand_name",
    "loves_count", "rating", "reviews", "size",
    "variation_type", "variation_value", "variation_desc",
    "ingredients", "price_usd", "value_price_usd", "sale_price_usd",
    "limited_edition", "new", "online_only", "out_of_stock",
    "sephora_exclusive", "highlights",
    "primary_category", "secondary_category", "tertiary_category",
    "child_count", "child_max_price", "child_min_price"
]

reviews = reviews[review_cols].copy()
products = products[product_cols].copy()

In [8]:
# Drop unnecessary index column if exists
if "Unnamed: 0" in reviews.columns:
    reviews = reviews.drop(columns=["Unnamed: 0"])

## 3. Avoid Column Conflicts

Rename product-side columns BEFORE merge
to avoid _x / _y confusion.

In [9]:
products = products.rename(columns={
    "product_name": "product_name_meta",
    "brand_name": "brand_name_meta",
    "rating": "product_rating",
    "reviews": "product_review_count",
    "price_usd": "price_usd_meta"
})

## 4. Review-Centric Merge

We use LEFT JOIN:
- keep all reviews
- attach product metadata

In [10]:
df = reviews.merge(products, on="product_id", how="left")
print(df.shape)
df.head()

(1094411, 44)


,author_id,rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,review_title,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd,product_name_meta,brand_id,brand_name_meta,loves_count,product_rating,product_review_count,size,variation_type,variation_value,variation_desc,ingredients,price_usd_meta,value_price_usd,sale_price_usd,limited_edition,new,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
0,1741593524,5,1.0,1.0,2,0,2,2023-02-01,I use this with the Nudestix “Citrus Clean Balm & Make-Up Melt“ to double cleanse and it has completely changed my skin (for the better). The make-up melt is oil based and removes all of your make...,Taught me how to double cleanse!,NaN,brown,dry,black,P504322,Gentle Hydra-Gel Face Cleanser,NUDESTIX,19.0,Gentle Hydra-Gel Face Cleanser,7055,NUDESTIX,177,5.0000,1.0,2.4 oz / 70 ml,Size,2.4 oz / 70 ml,NaN,"['Water (Aqua), Dipropylene Glycol, Peg-6 Caprylic/Capric Glycerides, Glycerin, 1,2-Hexanediol, Polyglyceryl-4 Caprate, Butylene Glycol, Carbomer, Propanediol, Tromethamine, Peg-60 Hydrogenated Ca...",19.0,NaN,NaN,0,0,1,0,0,['Clean at Sephora'],Skincare,Cleansers,NaN,0,NaN,NaN
1,31423088263,1,0.0,NaN,0,0,0,2023-03-21,"I bought this lip mask after reading the reviews and the hype. Unfortunately, it did not meet my expectations as vaseline petroleum jelly works way better for me.",Disappointed,NaN,NaN,NaN,NaN,P420652,Lip Sleeping Mask Intense Hydration with Vitamin C,LANEIGE,24.0,Lip Sleeping Mask Intense Hydration with Vitamin C,6125,LANEIGE,1081315,4.3508,16118.0,0.7 oz/ 20 g,Color,Original,NaN,"['Diisostearyl Malate, Hydrogenated Polyisobutene, Phyto- Steryl/Isostearyl/Cetyl/Stearyl/Behenyl Dimer Dilinoleate, Hydrogenated Poly(C6-14 Olefin), Polybutene, Microcrystalline Wax / Cera Microc...",24.0,NaN,NaN,0,0,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'Community Favorite', 'Vitamin C', 'Hydrating', 'Good for: Dryness', 'Without Parabens']",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
2,5061282401,5,1.0,NaN,0,0,0,2023-03-21,My review title says it all! I get so excited to get into bed and apply this lip mask. I do see a difference because I suffer from dry cracked lips. I drink a lot of water and apply lip balm daily...,New Favorite Routine,light,brown,dry,blonde,P420652,Lip Sleeping Mask Intense Hydration with Vitamin C,LANEIGE,24.0,Lip Sleeping Mask Intense Hydration with Vitamin C,6125,LANEIGE,1081315,4.3508,16118.0,0.7 oz/ 20 g,Color,Original,NaN,"['Diisostearyl Malate, Hydrogenated Polyisobutene, Phyto- Steryl/Isostearyl/Cetyl/Stearyl/Behenyl Dimer Dilinoleate, Hydrogenated Poly(C6-14 Olefin), Polybutene, Microcrystalline Wax / Cera Microc...",24.0,NaN,NaN,0,0,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'Community Favorite', 'Vitamin C', 'Hydrating', 'Good for: Dryness', 'Without Parabens']",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
3,6083038851,5,1.0,NaN,0,0,0,2023-03-20,I’ve always loved this formula for a long time. I honestly don’t even use it for night time. I use it as an everyday lip balm. I love the texture. Gummy Bear is my second most favourite scent. Gra...,Can't go wrong with any of them,NaN,brown,combination,black,P420652,Lip Sleeping Mask Intense Hydration with Vitamin C,LANEIGE,24.0,Lip Sleeping Mask Intense Hydration with Vitamin C,6125,LANEIGE,1081315,4.3508,16118.0,0.7 oz/ 20 g,Color,Original,NaN,"['Diisostearyl Malate, Hydrogenated Polyisobutene, Phyto- Steryl/Isostearyl/Cetyl/Stearyl/Behenyl Dimer Dilinoleate, Hydrogenated Poly(C6-14 Olefin), Polybutene, Microcrystalline Wax / Cera Microc...",24.0,NaN,NaN,0,0,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'Community Favorite', 'Vitamin C', 'Hydrating', 'Good for: Dryness', 'Without Parabens']",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
4,47056667835,5,1.0,NaN,0,0,0,2023-03-20,"If 

In [11]:
print("Null product_id:", df["product_id"].isna().sum())
print("Null review_text:", df["review_text"].isna().sum())
print("Null review rating:", df["rating"].isna().sum())
print("Null product_name (review side):", df["product_name"].isna().sum())
print("Null product_name_meta (product side):", df["product_name_meta"].isna().sum())
print("Null brand_name_meta:", df["brand_name_meta"].isna().sum())

Null product_id: 0
Null review_text: 1444
Null review rating: 0
Null product_name (review side): 0
Null product_name_meta (product side): 0
Null brand_name_meta: 0


In [12]:
df[[
    "product_id",
    "product_name",
    "product_name_meta",
    "brand_name",
    "brand_name_meta",
    "price_usd",
    "price_usd_meta"
]].sample(10, random_state=42)

,product_id,product_name,product_name_meta,brand_name,brand_name_meta,price_usd,price_usd_meta
1049894,P470240,Truth Serum Vitamin C Serum,Truth Serum Vitamin C Serum,OLEHENRIKSEN,OLEHENRIKSEN,56.0,56.0
619554,P423259,Cicapair Tiger Grass Serum,Cicapair Tiger Grass Serum,Dr. Jart+,Dr. Jart+,52.0,52.0
634782,P449175,Aqua Bomb Hydrating Toner,Aqua Bomb Hydrating Toner,belif,belif,30.0,30.0
706815,P4015,Clarifying Toner,Clarifying Toner,Murad,Murad,30.0,30.0
731656,P464233,Mini Face Cream,Mini Face Cream,Dr. Barbara Sturm,Dr. Barbara Sturm,75.0,75.0
37430,P248407,Ultra Repair Cream Intense Hydration,Ultra Repair Cream Intense Hydration,First Aid Beauty,First Aid Beauty,38.0,38.0
845174,P415620,Evercalm Ultra Comforting Rescue Mask,Evercalm Ultra Comforting Rescue Mask,REN Clean Skincare,REN Clean Skincare,52.0,52.0
570505,P434545,Glow Cycle Retin-ALT Power Serum,Glow Cycle Retin-ALT Power Serum,OLEHENRIKSEN,OLEHENRIKSEN,60.0,60.0
347448,P266126,Rosebud Salve in a Tube,Rosebud Salve in a Tube,Rosebud Perfume Co.,Rosebud Perfume Co.,7.0,7.0
672831,P404793,Sunshine Vitamin C + Squalane Face Oil,Sunshine Vitamin C + Squalane Face Oil,MILK MAKEUP,MILK MAKEUP,39.0,39.0


## 5. Canonical Columns

Create unified final columns for product name, brand name, and price.
Product metadata is prioritized; if it is missing, review-side values are used.

In [13]:
df["product_name_final"] = df["product_name_meta"].fillna(df["product_name"])
df["brand_name_final"] = df["brand_name_meta"].fillna(df["brand_name"])
df["price_usd_final"] = df["price_usd_meta"].fillna(df["price_usd"])

## 6. Data Type Fixing

In [14]:
numeric_cols = [
    "rating", "is_recommended", "helpfulness",
    "total_feedback_count", "total_neg_feedback_count",
    "total_pos_feedback_count",
    "price_usd", "price_usd_meta", "price_usd_final",
    "brand_id", "loves_count", "product_rating", "product_review_count"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [15]:
df["submission_time"] = pd.to_datetime(df["submission_time"], errors="coerce")

In [16]:
binary_cols = [
    "limited_edition",
    "new",
    "online_only",
    "out_of_stock",
    "sephora_exclusive"
]

for col in binary_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("Int64")

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1094411 entries, 0 to 1094410
Data columns (total 47 columns):
 #   Column                    Non-Null Count    Dtype         
---  ------                    --------------    -----         
 0   author_id                 1094411 non-null  string        
 1   rating                    1094411 non-null  int64         
 2   is_recommended            926423 non-null   float64       
 3   helpfulness               532819 non-null   float64       
 4   total_feedback_count      1094411 non-null  int64         
 5   total_neg_feedback_count  1094411 non-null  int64         
 6   total_pos_feedback_count  1094411 non-null  int64         
 7   submission_time           1094411 non-null  datetime64[ns]
 8   review_text               1092967 non-null  object        
 9   review_title              783757 non-null   object        
 10  skin_tone                 923872 non-null   object        
 11  eye_color                 884783 non-null   object

## 7. Handle Missing Values

We only drop rows missing critical fields:
- product_id
- review_text
- rating

In [18]:
null_summary = (
    df.isnull()
      .sum()
      .sort_values(ascending=False)
      .to_frame("null_count")
)

null_summary["null_ratio"] = null_summary["null_count"] / len(df)
null_summary.head(20)

,null_count,null_ratio
variation_desc,1086128,0.992432
sale_price_usd,1084658,0.991088
value_price_usd,1063534,0.971787
child_min_price,641008,0.585710
child_max_price,641008,0.585710
helpfulness,561592,0.513145
review_title,310654,0.283855
hair_color,226768,0.207206
eye_color,209628,0.191544
skin_tone,170539,0.155827


In [ ]:
# Drop `variation_desc`, `sale_price_usd`,`value_price_usd`.
# These columns are almost entirely missing and currently have low value for the first modeling iteration.

drop_cols = [
    "variation_desc",
    "sale_price_usd",
    "value_price_usd"
]

df = df.drop(columns=drop_cols)

In [ ]:
# For review-level modeling, these fields are essential: product_id, review_text, rating.
# Only rows missing these core fields will be removed.
df = df.dropna(subset=["product_id", "review_text", "rating"]).copy()

## 8. Remove Duplicates

In [21]:
print("Duplicates:", df.duplicated().sum())
df = df.drop_duplicates()

Duplicates: 224


## 9. Preserve Raw Text

We DO NOT clean text here.
We only create a combined raw_text field.

In [22]:
df["review_title"] = df["review_title"].fillna("").astype(str)
df["review_text"] = df["review_text"].fillna("").astype(str)

df["raw_text"] = (
    df["review_title"].str.strip() + " " +
    df["review_text"].str.strip()
).str.strip()

In [23]:
df[["review_title", "review_text", "raw_text"]].sample(5, random_state=42)

,review_title,review_text,raw_text
575159,,"Gifted by Dermalogica, I’ve been using this precleanse oil for the longest time. It helps me make sure that all the gunk and oil residue from the day breaks down and makes the perfect supporting p...","Gifted by Dermalogica, I’ve been using this precleanse oil for the longest time. It helps me make sure that all the gunk and oil residue from the day breaks down and makes the perfect supporting p..."
8582,,Smells good and has a nice consistency but burned my lips.,Smells good and has a nice consistency but burned my lips.
367561,Can't Go Wrong...,Went back to Clinique after using Murad because it dried out my skin so badly. I have used Clinique for more years than I can count at this point. In one wash it cleared up the breakouts and dryne...,Can't Go Wrong... Went back to Clinique after using Murad because it dried out my skin so badly. I have used Clinique for more years than I can count at this point. In one wash it cleared up the b...
982229,Worth the price.,Probably the only eye cream that made a difference to my under eye darkness. Plan to purchase again.,Worth the price. Probably the only eye cream that made a difference to my under eye darkness. Plan to purchase again.
283247,Nice cream,"I received this product complimentary from Influenster for my honest review. This is super hydrating without leaving skin feelingsticky or like it clogged your pores, I don’t have anybreakout issu...","Nice cream I received this product complimentary from Influenster for my honest review. This is super hydrating without leaving skin feelingsticky or like it clogged your pores, I don’t have anybr..."


## 10. Basic Text Features

In [24]:
df["review_text_length"] = df["review_text"].str.len()
df["raw_text_length"] = df["raw_text"].str.len()
df["has_title"] = (df["review_title"].str.len() > 0).astype(int)

## 11. Rating Category (Helper Label)

Used only for baseline sentiment modeling later.

In [25]:
def rating_cat(x):
    if x >= 4:
        return "positive"
    elif x == 3:
        return "neutral"
    else:
        return "negative"

df["rating_category"] = df["rating"].apply(rating_cat)
df["rating_category"].value_counts(dropna=False)

rating_category
positive    896943
negative    114051
neutral      81749
Name: count, dtype: int64

## 12. Create Final Tables

In [31]:
review_master = df[[
    "author_id",
    "product_id",
    "product_name_final",
    "brand_name_final",
    "rating",
    "rating_category",
    "is_recommended",
    "helpfulness",
    "total_feedback_count",
    "total_neg_feedback_count",
    "total_pos_feedback_count",
    "submission_time",
    "review_title",
    "review_text",
    "raw_text",
    "review_text_length",
    "raw_text_length",
    "has_title",
    "skin_tone",
    "eye_color",
    "skin_type",
    "hair_color",
    "price_usd_final",
    "brand_id",
    "loves_count",
    "product_rating",
    "product_review_count",
    "size",
    "variation_type",
    "variation_value",
    "ingredients",
    "limited_edition",
    "new",
    "online_only",
    "out_of_stock",
    "sephora_exclusive",
    "highlights",
    "primary_category",
    "secondary_category",
    "tertiary_category",
    "child_count",
    "child_max_price",
    "child_min_price"
]].copy()

In [32]:
print("review_master shape:", review_master.shape)
review_master.head()

review_master shape: (1092743, 43)


,author_id,product_id,product_name_final,brand_name_final,rating,rating_category,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_title,review_text,raw_text,review_text_length,raw_text_length,has_title,skin_tone,eye_color,skin_type,hair_color,price_usd_final,brand_id,loves_count,product_rating,product_review_count,size,variation_type,variation_value,ingredients,limited_edition,new,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
0,1741593524,P504322,Gentle Hydra-Gel Face Cleanser,NUDESTIX,5,positive,1.0,1.0,2,0,2,2023-02-01,Taught me how to double cleanse!,I use this with the Nudestix “Citrus Clean Balm & Make-Up Melt“ to double cleanse and it has completely changed my skin (for the better). The make-up melt is oil based and removes all of your make...,Taught me how to double cleanse! I use this with the Nudestix “Citrus Clean Balm & Make-Up Melt“ to double cleanse and it has completely changed my skin (for the better). The make-up melt is oil b...,455,488,1,NaN,brown,dry,black,19.0,7055,177,5.0000,1.0,2.4 oz / 70 ml,Size,2.4 oz / 70 ml,"['Water (Aqua), Dipropylene Glycol, Peg-6 Caprylic/Capric Glycerides, Glycerin, 1,2-Hexanediol, Polyglyceryl-4 Caprate, Butylene Glycol, Carbomer, Propanediol, Tromethamine, Peg-60 Hydrogenated Ca...",0,0,1,0,0,['Clean at Sephora'],Skincare,Cleansers,NaN,0,NaN,NaN
1,31423088263,P420652,Lip Sleeping Mask Intense Hydration with Vitamin C,LANEIGE,1,negative,0.0,NaN,0,0,0,2023-03-21,Disappointed,"I bought this lip mask after reading the reviews and the hype. Unfortunately, it did not meet my expectations as vaseline petroleum jelly works way better for me.","Disappointed I bought this lip mask after reading the reviews and the hype. Unfortunately, it did not meet my expectations as vaseline petroleum jelly works way better for me.",162,175,1,NaN,NaN,NaN,NaN,24.0,6125,1081315,4.3508,16118.0,0.7 oz/ 20 g,Color,Original,"['Diisostearyl Malate, Hydrogenated Polyisobutene, Phyto- Steryl/Isostearyl/Cetyl/Stearyl/Behenyl Dimer Dilinoleate, Hydrogenated Poly(C6-14 Olefin), Polybutene, Microcrystalline Wax / Cera Microc...",0,0,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'Community Favorite', 'Vitamin C', 'Hydrating', 'Good for: Dryness', 'Without Parabens']",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
2,5061282401,P420652,Lip Sleeping Mask Intense Hydration with Vitamin C,LANEIGE,5,positive,1.0,NaN,0,0,0,2023-03-21,New Favorite Routine,My review title says it all! I get so excited to get into bed and apply this lip mask. I do see a difference because I suffer from dry cracked lips. I drink a lot of water and apply lip balm daily...,New Favorite Routine My review title says it all! I get so excited to get into bed and apply this lip mask. I do see a difference because I suffer from dry cracked lips. I drink a lot of water and...,272,293,1,light,brown,dry,blonde,24.0,6125,1081315,4.3508,16118.0,0.7 oz/ 20 g,Color,Original,"['Diisostearyl Malate, Hydrogenated Polyisobutene, Phyto- Steryl/Isostearyl/Cetyl/Stearyl/Behenyl Dimer Dilinoleate, Hydrogenated Poly(C6-14 Olefin), Polybutene, Microcrystalline Wax / Cera Microc...",0,0,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'Community Favorite', 'Vitamin C', 'Hydrating', 'Good for: Dryness', 'Without Parabens']",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
3,6083038851,P420652,Lip Sleeping Mask Intense Hydration with Vitamin C,LANEIGE,5,positive,1.0,NaN,0,0,0,2023-03-20,Can't go wrong with any of them,I’ve always loved this formula for a long time. I honestly don’t even use it for night time. I use it as an everyday lip balm. I love the texture. Gummy Bear is my second most favourite scent. Gra...,Can't go wrong with any of them I’ve always loved this formula for a long time. I honestly don’t even use it for night time. I use it as an everyday lip balm. I love the texture

In [33]:
product_master = products.copy()

product_master["product_name_final"] = product_master["product_name_meta"]
product_master["brand_name_final"] = product_master["brand_name_meta"]
product_master["price_usd_final"] = product_master["price_usd_meta"]

In [34]:
print("product_master shape:", product_master.shape)
product_master.head()

product_master shape: (8494, 30)


,product_id,product_name_meta,brand_id,brand_name_meta,loves_count,product_rating,product_review_count,size,variation_type,variation_value,variation_desc,ingredients,price_usd_meta,value_price_usd,sale_price_usd,limited_edition,new,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price,product_name_final,brand_name_final,price_usd_final
0,P473671,Fragrance Discovery Set,6342,19-69,6320,3.6364,11.0,NaN,NaN,NaN,NaN,"['Capri Eau de Parfum:', 'Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) D-Limonene, Linalool, Benzyl Salicylate, Ethylhexyl Methoxycinnamate, Butyl Methoxydibenzoylmethane, Ethylhexyl Salicy...",35.0,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Warm &Spicy Scent', 'Woody & Earthy Scent', 'Fresh Scent']",Fragrance,Value & Gift Sets,Perfume Gift Sets,0,NaN,NaN,Fragrance Discovery Set,19-69,35.0
1,P473668,La Habana Eau de Parfum,6342,19-69,3827,4.1538,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,NaN,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) Ethylhexyl Methoxycinnamate, Ethylhexyl Salicylate, Butyl Methoxydibenzoylmethane, Benzyl Alcohol, Benzyl Benzoate, Benzyl Cinnamate, Cinnamal...",195.0,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent', 'Warm &Spicy Scent']",Fragrance,Women,Perfume,2,85.0,30.0,La Habana Eau de Parfum,19-69,195.0
2,P473662,Rainbow Bar Eau de Parfum,6342,19-69,3253,4.2500,16.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,NaN,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) D-Limonene, Ethylhexyl Methoxycinnamate, Butyl Methoxydibenzoylmethane, Ethylhexyl Salicylate, Hexyl Cinnamal Citronellol, Linalool, Coumarin,...",195.0,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent', 'Woody & Earthy Scent']",Fragrance,Women,Perfume,2,75.0,30.0,Rainbow Bar Eau de Parfum,19-69,195.0
3,P473660,Kasbah Eau de Parfum,6342,19-69,3018,4.4762,21.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,NaN,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) Coumarin, Ethylhexyl Methoxycinnamate, Butyl Methoxydibenzoylmethane, Ethylhexyl Salicylate, D-Limonene, Eugenol, Linalool, Citronellol, Geran...",195.0,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent', 'Warm &Spicy Scent']",Fragrance,Women,Perfume,2,75.0,30.0,Kasbah Eau de Parfum,19-69,195.0
4,P473658,Purple Haze Eau de Parfum,6342,19-69,2691,3.2308,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,NaN,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) Octoxynol-10, D-Limonene, Ethylhexyl Methoxycinnamate, Butyl Methoxydibenzoylmethane, Ethylhexyl Salicylate, Geraniol, Linalool, Coumarin, Far...",195.0,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent', 'Woody & Earthy Scent']",Fragrance,Women,Perfume,2,75.0,30.0,Purple Haze Eau de Parfum,19-69,195.0


## 13. Quality Checks

In [35]:
print(review_master.shape)
print(product_master.shape)

print("\nRating category distribution:")
print(review_master["rating_category"].value_counts(dropna=False))

print("\nPrimary category distribution:")
print(review_master["primary_category"].value_counts(dropna=False).head(10))

print("\nSkin type distribution:")
print(review_master["skin_type"].value_counts(dropna=False).head(10))

(1092743, 43)
(8494, 30)

Rating category distribution:
rating_category
positive    896943
negative    114051
neutral      81749
Name: count, dtype: int64

Primary category distribution:
primary_category
Skincare    1092743
Name: count, dtype: int64

Skin type distribution:
skin_type
combination    543712
dry            185651
normal         131674
oily           120296
NaN            111410
Name: count, dtype: int64


## 14. Save Outputs

In [36]:
review_master.to_parquet(PROCESSED_DIR / "review_master.parquet", index=False)
product_master.to_parquet(PROCESSED_DIR / "product_master.parquet", index=False)

print("Files saved successfully.")

Files saved successfully.


# Conclusion

This notebook produced:
- a clean review-level master table
- a clean product-level metadata table
- preserved raw review text
- a controlled missing-value strategy
- a basic dataset bias assessment

The next notebook will focus on:
- text preprocessing
- concern dictionary design
- typo/variant-tolerant matching
- review-to-concern mapping